# 02 — Multi-source Preprocessing for Conversational NLP

**Project context**: this notebook belongs to the chatbot layer of the Customer Behaviour Analysis project.

**Final model outputs**:
- `intent`: what the customer is asking for.


**Initial issue**: training only on `bitext_retail` gives strong in-domain results but weak external generalization.  
The model learns the Bitext writing style instead of learning reusable intent semantics.

**Solution**: build a multi-source preprocessing pipeline using Bitext, support datasets, synthetic e-commerce data, FAQ intents, and Amazon QA sources.  
The pipeline standardizes schemas, maps labels, filters noisy intents, balances sources, and creates source-aware train/validation/test splits.

## Pipeline
1. Imports and configuration
2. Preprocessing decisions
3. Schema mapping utilities
4. Text cleaning and label normalization
5. Dataset loaders
6. Dataset standardization
7. Amazon QA heuristic intent mapping
8. Loading and standardizing all sources
9. Intent consolidation
10. Canonical intent space definition
11. Canonical filtering
12. Merge, deduplication, and label encoding
13. Source capping
14. Source-aware train/validation/test split
15. Light train-only augmentation
16. Final split validation
17. Saving the splits
18. Final summary


## 1. Imports and configuration


In [1]:
from pathlib import Path
import json
import re
import random
import unicodedata
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_DIR  = Path("D:/conv_nlp_pipeline")
DATA_RAW     = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
DATA_SPLITS  = PROJECT_DIR / "data" / "splits"
REPORTS_DIR  = PROJECT_DIR / "reports" / "preprocessing"

for p in [DATA_PROCESSED, DATA_SPLITS, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_RAW      :", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("DATA_SPLITS   :", DATA_SPLITS)
print("REPORTS_DIR   :", REPORTS_DIR)

DATA_RAW      : D:\conv_nlp_pipeline\data\raw
DATA_PROCESSED: D:\conv_nlp_pipeline\data\processed
DATA_SPLITS   : D:\conv_nlp_pipeline\data\splits
REPORTS_DIR   : D:\conv_nlp_pipeline\reports\preprocessing


## 2. Preprocessing decisions

These decisions come from `01_exploration.ipynb`.  
They explain why each source is used and how datasets without native intents, such as Amazon QA, are handled.


In [2]:
PREPROCESSING_DECISIONS = {
    "project_layer"      : "chatbot_conversation_analysis",
    "final_outputs"     : ["intent", "sentiment", "churn_risk"],
    "main_reference"    : "bitext-retail.csv",
    "accepted_sources"  : [
        "bitext_retail",          # main reference source, explicit intents
        "bitext_support",         # customer support source, explicit intents
        "synthetic_ecommerce",    # synthetic e-commerce source, explicit intents
        "faq_intents",            # chatbot FAQ source, explicit intents
        "amazon_single_qna",      # Amazon product QA, heuristic intent mapping
        "amazon_multi_questions", # Amazon multi-question QA, heuristic intent mapping
    ],
    "problem"           : "bitext alone does not generalize well to external customer messages",
    "amazon_strategy"   : "heuristic_mapping_on_question_text_ignore_product_category",
    "generalization_goal": "train on varied writing styles while keeping a clean canonical intent space",
    "rejected"          : [
        "Amazon Category is a product type, not a customer intent",
        "samples with intent=unknown after heuristic mapping",
        "intents outside canonical_set after consolidation",
    ],
}

pd.DataFrame(
    {"decision": PREPROCESSING_DECISIONS.keys(), "value": PREPROCESSING_DECISIONS.values()}
)


,decision,value
0,project_layer,chatbot_conversation_analysis
1,final_outputs,"[intent, sentiment, churn_risk]"
2,main_reference,bitext-retail.csv
3,accepted_sources,"[bitext_retail, bitext_support, synthetic_ecom..."
4,problem,bitext alone does not generalize well to exter...
5,amazon_strategy,heuristic_mapping_on_question_text_ignore_prod...
6,generalization_goal,train on varied writing styles while keeping a...
7,rejected,"[Amazon Category is a product type, not a cust..."


## 3. Schema mapping utilities

Each dataset uses different column names such as `instruction`, `text`, `question`, `intent`, `label`, or `tag`.  
These utilities automatically detect the relevant columns and reduce hard-coding.


In [3]:
def normalize_col_name(s: str) -> str:
    s = str(s).strip().lower()
    s = "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")


TEXT_COL_CANDIDATES = [
    "instruction", "text", "utterance", "question", "input", "query", "sentence",
    "prompt", "user_message", "customer_query", "message", "request",
    "Question", "QuestionText",
    "requete", "demande", "question_client", "texte",
]

RESPONSE_COL_CANDIDATES = [
    "response", "answer", "output", "completion", "assistant_response",
    "agent_response", "Answer",
    "reponse", "réponse", "reponse_agent",
]

INTENT_COL_CANDIDATES = [
    "intent", "label", "tag", "class", "intent_name", "category_label",
    "intention", "classe", "etiquette",
]

CATEGORY_COL_CANDIDATES = [
    "category", "domain", "group", "Category", "section", "topic",
    "categorie", "catégorie", "domaine",
]

FLAGS_COL_CANDIDATES = ["flags", "tags", "metadata", "meta", "notes"]


def find_best_existing_col(df: pd.DataFrame, candidates: list) -> str | None:
    if df is None or df.empty:
        return None
    norm_to_original = {normalize_col_name(c): c for c in df.columns}
    norm_candidates  = [normalize_col_name(c) for c in candidates]
    for cand in norm_candidates:
        if cand in norm_to_original:
            return norm_to_original[cand]
    for norm_col, original_col in norm_to_original.items():
        for cand in norm_candidates:
            if cand and cand in norm_col:
                return original_col
    return None


def infer_schema_mapping(df: pd.DataFrame) -> dict:
    return {
        "text_col"    : find_best_existing_col(df, TEXT_COL_CANDIDATES),
        "response_col": find_best_existing_col(df, RESPONSE_COL_CANDIDATES),
        "intent_col"  : find_best_existing_col(df, INTENT_COL_CANDIDATES),
        "category_col": find_best_existing_col(df, CATEGORY_COL_CANDIDATES),
        "flags_col"   : find_best_existing_col(df, FLAGS_COL_CANDIDATES),
    }

## 4. Text cleaning and label normalization

- `clean_text`: normalizes Bitext placeholders such as `{{Order Number}}` into `[ORDER]`.
- `normalize_label`: creates stable snake_case labels.
- `INTENT_ALIASES`: maps alternative labels to canonical labels.
- `canonicalize_intent`: applies normalization and alias mapping in one pass.


In [4]:
PLACEHOLDER_MAP = {
    r"\{\{\s*Order Number\s*\}\}"  : "[ORDER]",
    r"\{\{\s*Order ID\s*\}\}"      : "[ORDER]",
    r"\{\{\s*Product Name\s*\}\}"  : "[PRODUCT]",
    r"\{\{\s*Item Name\s*\}\}"     : "[ITEM]",
    r"\{\{\s*Customer Name\s*\}\}": "[CUSTOMER]",
    r"\{\{\s*Email\s*\}\}"         : "[EMAIL]",
    r"\{\{\s*Phone Number\s*\}\}" : "[PHONE]",
    r"\{\{\s*Store Location\s*\}\}": "[STORE]",
    r"\{\{\s*Website URL\s*\}\}"  : "[URL]",
}


def clean_text(text: str) -> str:
    if pd.isna(text): return ""
    text = str(text)
    
    # existing bitext placeholders
    for pattern, replacement in PLACEHOLDER_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    
    # ADD: normalize real-world entity formats
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '[EMAIL]', text)
    text = re.sub(r'\b(?:order|ord|ref|#)\s*[-_]?\s*\d{4,}\b', '[ORDER]', text, flags=re.I)
    text = re.sub(r'\b\d{5,}\b', '[NUMBER]', text)  # bare long numbers
    text = re.sub(r'\$\s?\d+[\d,.]*|\d+[\d,.]*\s?(?:usd|eur|gbp|tnd|dt)', '[PRICE]', text, flags=re.I)
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', '[DATE]', text)
    text = re.sub(r'\b(?:\+?\d[\d\s\-().]{7,}\d)\b', '[PHONE]', text)
    
    text = text.replace('\n', ' ').replace('\t', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_label(label: str) -> str:
    if pd.isna(label):
        return "unknown"
    label = str(label).strip().lower()
    label = "".join(
        c for c in unicodedata.normalize("NFD", label)
        if unicodedata.category(c) != "Mn"
    )
    label = re.sub(r"[^a-z0-9]+", "_", label)
    label = re.sub(r"_+", "_", label).strip("_")
    return label if label else "unknown"


INTENT_ALIASES = {
    # Refund / return
    "refund"               : "request_refund",
    "get_refund"           : "request_refund",
    "refund_request"       : "request_refund",
    "return"               : "return_product",
    "return_item"          : "return_product",
    "return_request"       : "return_product",
    # Tracking
    "where_is_my_order"    : "track_order",
    "order_tracking"       : "track_order",
    "track_my_order"       : "track_order",
    "shipping_status"      : "track_delivery",
    "delivery_status"      : "track_delivery",
    "track_shipping"       : "track_delivery",
    "track_package"        : "track_delivery",
    "package_tracking"     : "track_delivery",
    # Payment
    "checkout"             : "pay",
    "make_payment"         : "pay",
    "payment_problem"      : "payment_issue",
    # Product
    "product_info"         : "product_information",
    "item_information"     : "product_information",
    "product_detail"       : "product_information",
    "product_question"     : "product_information",
    "faq_product"          : "product_information",
    "product_availability" : "availability",
    "stock_check"          : "availability",
    "in_stock"             : "availability",
    # Support
    "contact"              : "contact_human_agent",
    "support"              : "contact_human_agent",
    "agent"                : "contact_human_agent",
}


def canonicalize_intent(intent: str) -> str:
    intent = normalize_label(intent)
    return INTENT_ALIASES.get(intent, intent)

## 5. Dataset loaders

Robust loaders return an empty DataFrame if an optional file does not exist.  
This prevents the whole pipeline from failing when one optional source is missing.


In [5]:
def read_csv_if_exists(path: Path, **kwargs) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path, **kwargs)
    print(f"[MISSING] {path}")
    return pd.DataFrame()


def read_json_intents(path: Path) -> pd.DataFrame:
    """Loads classic FAQ JSON structures (patterns/responses) into a flat DataFrame."""
    if not path.exists():
        print(f"[MISSING] {path}")
        return pd.DataFrame()
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    rows = []
    intents = data.get("intents", data if isinstance(data, list) else [])
    for item in intents:
        if not isinstance(item, dict):
            continue
        tag       = item.get("tag") or item.get("intent") or item.get("label") or item.get("category")
        category  = item.get("category") or item.get("domain") or "faq"
        patterns  = item.get("patterns") or item.get("questions") or item.get("examples") or []
        responses = item.get("responses") or item.get("answers") or [""]
        response  = responses[0] if isinstance(responses, list) and responses else str(responses)
        for p in patterns:
            rows.append({"question": p, "answer": response, "tag": tag, "category": category})
    return pd.DataFrame(rows)


# Chemins vers les fichiers bruts — adapter si nécessaire
RAW_PATHS = {
    "bitext_retail"         : DATA_RAW / "bitext-retail.csv",
    "bitext_support"        : DATA_RAW / "Bitext_Customer_Support.csv",
    "synthetic_ecommerce"   : DATA_RAW / "synthetic_ecommerce_data.csv",
    "faq_intents"           : DATA_RAW / "Ecommerce_FAQ_intents.json",
    "amazon_single_qna"     : DATA_RAW / "single_qna.csv",
    "amazon_multi_questions": DATA_RAW / "multi_questions.csv",
}

## 6. Dataset standardization

Converts any raw dataset to the common schema:

`instruction | response | intent | category | flags | source | mapping_method`

Rules:
- If an intent column exists, the sample uses `explicit_intent`.
- If no intent column exists and `use_heuristic_if_no_intent=True`, a heuristic mapping is used.
- If `require_intent=True` and no intent can be found, the source is skipped.


In [6]:
def standardize_dataset(
    df: pd.DataFrame,
    source_name: str,
    require_intent: bool = True,
    use_heuristic_if_no_intent: bool = False,
) -> pd.DataFrame:
    """Converts a raw dataset to the common preprocessing schema."""
    if df is None or df.empty:
        return pd.DataFrame()

    mapping      = infer_schema_mapping(df)
    text_col     = mapping["text_col"]
    response_col = mapping["response_col"]
    intent_col   = mapping["intent_col"]
    category_col = mapping["category_col"]
    flags_col    = mapping["flags_col"]

    if text_col is None:
        print(f"[SKIP] {source_name}: no text column found. Columns={df.columns.tolist()}")
        return pd.DataFrame()

    out = pd.DataFrame()
    out["instruction"] = df[text_col].apply(clean_text)
    out["response"]    = df[response_col].apply(clean_text) if response_col else ""
    out["category"]    = df[category_col].apply(normalize_label) if category_col else "unknown"
    out["flags"]       = df[flags_col].astype(str) if flags_col else ""
    out["source"]      = source_name

    if intent_col:
        out["intent"]          = df[intent_col].apply(canonicalize_intent)
        out["mapping_method"]  = "explicit_intent"
    elif use_heuristic_if_no_intent:
        out["intent"]          = out["instruction"].apply(heuristic_intent_from_text)
        out["mapping_method"]  = "heuristic"
    elif require_intent:
        print(f"[SKIP] {source_name}: pas de colonne intent et require_intent=True")
        return pd.DataFrame()
    else:
        out["intent"]         = "unknown"
        out["mapping_method"] = "unknown"

    out["instruction"] = out["instruction"].fillna("").astype(str)
    out["response"]    = out["response"].fillna("").astype(str)
    out = out[out["instruction"].str.len() > 0].copy()
    out = out[out["intent"].notna()].copy()

    out["word_count"]        = out["instruction"].str.split().str.len()
    out["char_count"]        = out["instruction"].str.len()
    out["has_placeholder"]   = out["instruction"].str.contains(r"\[[A-Z_]+\]", regex=True, na=False)
    out["placeholder_count"] = out["instruction"].str.count(r"\[[A-Z_]+\]")

    print(f"[OK] {source_name}: {len(out):,} rows | intent_col={intent_col}")
    return out.reset_index(drop=True)

## 7. Amazon QA heuristic intent mapping

Amazon QA does not provide native customer-service intents.  
Its `Category` column describes product types such as electronics or appliances, not user intents.

**Strategy**: ignore `Category`, read the question text, and apply regex rules ordered from the most specific to the most general.  
The first matching rule defines the intent.

These rules cover the main e-commerce intents that can be detected from keywords.


In [7]:
# Ordered rules: from the most specific to the most general
# First match = selected intent (exactly one match is not required)
HEURISTIC_RULES = [
    ("track_order",         [r"\bwhere.*order\b", r"\btrack.*order\b", r"\border.*status\b", r"\border.*where\b"]),
    ("track_delivery",      [r"\bdeliver", r"\bshipping\b", r"\bshipment\b", r"\bpackage\b", r"\bparcel\b"]),
    ("request_refund",      [r"\brefund\b", r"\bmoney back\b", r"\bget.*money\b"]),
    ("return_product",      [r"\breturn\b", r"\bsend.*back\b", r"\bsend it back\b"]),
    ("cancel_order",        [r"\bcancel\b"]),
    ("payment_issue",       [r"\bpayment.*(fail|error|issue|problem|declin)\b", r"\bcard.*declin", r"\bcharge.*wrong\b"]),
    ("pay",                 [r"\bpay\b", r"\bpayment\b", r"\bcheckout\b", r"\bcredit card\b", r"\bbilling\b"]),
    ("availability",        [r"\bin stock\b", r"\bavailable\b", r"\bavailability\b", r"\bout of stock\b", r"\bback in stock\b"]),
    ("product_information", [r"\bsize\b", r"\bcolor\b", r"\bcolour\b", r"\bmaterial\b", r"\bcompatible\b",
                              r"\bdimension\b", r"\bfeature\b", r"\bweight\b", r"\bfit\b", r"\bwork with\b",
                              r"\bhow does.*work\b", r"\bcome with\b", r"\binclude\b", r"\bmodel\b",
                              r"\bspecification\b", r"\bwatt\b", r"\bvolt\b", r"\binch\b", r"\bgallon\b"]),
    ("change_order",        [r"\bchange.*order\b", r"\bmodify.*order\b", r"\bupdate.*order\b", r"\bedit.*order\b"]),
    ("missing_item",        [r"\bmissing\b", r"\bnot.*receiv\b", r"\bnever.*arriv\b", r"\bdidn.*t.*come\b"]),
    ("damaged_delivery",    [r"\bdamage\b", r"\bbroken\b", r"\bdefect\b", r"\bnot.*work\b", r"\bstop.*work\b"]),
    ("delivery_time",       [r"\bhow long\b", r"\bwhen.*arriv\b", r"\bwhen.*deliver\b", r"\bestimate\b", r"\bdays?\b.*deliver"]),
    ("request_invoice",     [r"\binvoice\b", r"\breceipt\b", r"\bbill\b"]),
    ("create_account",      [r"\bcreate.*account\b", r"\bsign up\b", r"\bregister\b", r"\bnew account\b"]),
    ("change_account",      [r"\bchange.*account\b", r"\bupdate.*account\b", r"\bedit.*profile\b"]),
    ("recover_password",    [r"\bpassword\b", r"\bforgot\b", r"\breset\b", r"\bcan.*t.*log\b"]),
    ("contact_human_agent", [r"\bcustomer service\b", r"\bsupport\b", r"\bcontact\b", r"\bhelp\b",
                              r"\bhuman\b", r"\breal person\b", r"\bspeak.*agent\b", r"\btalk.*agent\b"]),
]


def heuristic_intent_from_text(text: str) -> str:
    """Retourne le premier intent qui matche dans les règles (ordre = priorité)."""
    text_norm = clean_text(text).lower()
    for intent, patterns in HEURISTIC_RULES:
        if any(re.search(p, text_norm) for p in patterns):
            return intent
    return "unknown"


# Verification rapide de la heuristique
test_questions = [
    ("Does this come with a power cord?",   "product_information"),
    ("What is the size of this product?",   "product_information"),
    ("Can I return this item?",             "return_product"),
    ("Where is my order?",                  "track_order"),
    ("How long does shipping take?",         "track_delivery"),
    ("model number",                        "product_information"),
    ("I need a refund for my purchase",     "request_refund"),
    ("Can I cancel my order?",              "cancel_order"),
]
print("Heuristic sanity check:")
for q, expected in test_questions:
    result = heuristic_intent_from_text(q)
    status = "✓" if result == expected else "✗"
    print(f"  {status} {result:30s} ← '{q}'")

Heuristic sanity check:
  ✓ product_information            ← 'Does this come with a power cord?'
  ✓ product_information            ← 'What is the size of this product?'
  ✓ return_product                 ← 'Can I return this item?'
  ✓ track_order                    ← 'Where is my order?'
  ✓ track_delivery                 ← 'How long does shipping take?'
  ✓ product_information            ← 'model number'
  ✓ request_refund                 ← 'I need a refund for my purchase'
  ✓ cancel_order                   ← 'Can I cancel my order?'


## 8. Loading and standardizing all sources

**Important Amazon rule**: the `Category` column is dropped before calling `standardize_dataset`.

Without this, the pipeline may incorrectly treat product categories as intents.  
After dropping it, Amazon samples use text-based heuristic intent mapping.


In [8]:
# Required main source
retail_raw = read_csv_if_exists(RAW_PATHS["bitext_retail"])
if retail_raw.empty:
    raise FileNotFoundError(f"Missing main source : {RAW_PATHS['bitext_retail']}")
df_retail = standardize_dataset(retail_raw, "bitext_retail", require_intent=True)

# Optional sources with explicit intents
support_raw = read_csv_if_exists(RAW_PATHS["bitext_support"])
synth_raw   = read_csv_if_exists(RAW_PATHS["synthetic_ecommerce"])
faq_raw     = read_json_intents(RAW_PATHS["faq_intents"])

df_support = standardize_dataset(support_raw, "bitext_support", require_intent=True)
df_synth   = standardize_dataset(synth_raw,   "synthetic_ecommerce", require_intent=True)
df_faq     = standardize_dataset(faq_raw,     "faq_intents", require_intent=True)

# Amazon: drop Category to force heuristic mapping on the text
amazon_single_raw = read_csv_if_exists(RAW_PATHS["amazon_single_qna"])
amazon_multi_raw  = read_csv_if_exists(RAW_PATHS["amazon_multi_questions"])

df_amazon_single = standardize_dataset(
    amazon_single_raw.drop(columns=["Category"], errors="ignore"),
    "amazon_single_qna",
    require_intent=False,
    use_heuristic_if_no_intent=True,
)
df_amazon_multi = standardize_dataset(
    amazon_multi_raw.drop(columns=["Category"], errors="ignore"),
    "amazon_multi_questions",
    require_intent=False,
    use_heuristic_if_no_intent=True,
)

all_sources = [df_retail, df_support, df_synth, df_faq, df_amazon_single, df_amazon_multi]
all_sources = [df for df in all_sources if df is not None and not df.empty]

print("\nSources chargées :")
for df in all_sources:
    src = df['source'].iloc[0]
    n_intents = df['intent'].nunique()
    print(f"  - {src:<25} {len(df):>8,} rows  |  {n_intents} intents uniques")

[OK] bitext_retail: 44,884 rows | intent_col=intent
[OK] bitext_support: 26,872 rows | intent_col=intent
[OK] synthetic_ecommerce: 1,005 rows | intent_col=intent
[OK] faq_intents: 422 rows | intent_col=tag
[OK] amazon_single_qna: 1,396,895 rows | intent_col=None
[OK] amazon_multi_questions: 172,617 rows | intent_col=None

Sources chargées :
  - bitext_retail               44,884 rows  |  46 intents uniques
  - bitext_support              26,872 rows  |  27 intents uniques
  - synthetic_ecommerce          1,005 rows  |  14 intents uniques
  - faq_intents                    422 rows  |  97 intents uniques
  - amazon_single_qna         1,396,895 rows  |  19 intents uniques
  - amazon_multi_questions     172,617 rows  |  18 intents uniques


## 9. Intent consolidation

Before filtering, semantically redundant intents are merged.

Examples:
- `delivery_period` and `delivery_time`
- `edit_account` and `change_account`

Intents mapped to `None` are removed because they are noisy or out of scope.


In [9]:
INTENT_MERGE_MAP = {
    # DELIVERY
    "delivery_period"            : "delivery_time",
    "delivery_issue"             : "track_delivery",
    "delivery_options"           : "track_delivery",
    "lost_or_damaged_package"    : "damaged_delivery",
    "international_shipping"     : "track_delivery",
    "multiple_shipping_addresses": "change_shipping_address",
    "in_store_pickup"            : "track_delivery",

    # RETURN / REFUND
    "return_policy"              : "check_refund_policy",
    "refund_policy"              : "check_refund_policy",
    "refund_status"              : "track_refund",
    "return_product_in_store"    : "return_product",
    "return_product_online"      : "return_product",
    "gift_returns"               : "return_product",
    "guest_returns"              : "return_product",
    "international_returns"      : "return_product",

    # ACCOUNT
    "open_account"               : "create_account",
    "account_deletion"           : "delete_account",
    "close_account"              : "delete_account",
    "edit_account"               : "change_account",
    "account_update_details"     : "change_account",
    "login_issues"               : "registration_problems",
    "guest_checkout"             : "create_account",

    # ORDER
    "order_cancellation_status"  : "cancel_order",
    "order_confirmation"         : "order_status",
    "order_fulfillment"          : "order_status",
    "order_issues_extra_item"    : "wrong_item",
    "modify_order"               : "change_order",
    "change_order_quantity"      : "change_order",
    "incorrect_item"             : "wrong_item",

    # INVOICE / PAYMENT
    "check_invoice"              : "request_invoice",
    "get_invoice"                : "request_invoice",
    "billing_issues"             : "payment_issue",
    "payment_methods"            : "check_payment_methods",
    "financing"                  : "check_payment_methods",
    "gift_card_purchase"         : "pay",
    "gift_cards"                 : "pay",

    # HUMAN / SUPPORT
    "contact_customer_service"   : "contact_human_agent",
    "customer_service"           : "contact_human_agent",
    "human_agent"                : "contact_human_agent",
    "customer_support"           : "contact_human_agent",
    "contact_department"         : "contact_human_agent",

    # PRODUCT
    "exchange_item"              : "exchange_product",
    "exchange_product_in_store"  : "exchange_product",
    "availability_in_store"      : "availability",
    "availability_online"        : "availability",
    "bulk_pricing"               : "pricing",
    "discounts_and_offers"       : "pricing",
    "coupon_application"         : "pricing",

    # FEEDBACK / NEWSLETTER
    "submit_product_feedback"    : "submit_feedback",
    "feedback"                   : "submit_feedback",
    "newsletter_benefits"        : "newsletter_subscription",
    "newsletter_unsubscription"  : "newsletter_subscription",
    "loyalty_tiers"              : "loyalty_program",

    # JUNK → None = supprimé du dataset
    "donkey_level_nonsense"      : None,
    "unknown"                    : None,
    "other"                      : None,
    "off_topic"                  : None,
    "faq"                        : None,
    "about_us"                   : None,
    "careers"                    : None,
    "corporate_responsibility"   : None,
    "eco_friendly_packaging"     : None,
    "dropshipping_partnerships"  : None,
    "affiliate_program"          : None,
    "b2b_sales"                  : None,
}


def apply_intent_merge(df: pd.DataFrame) -> pd.DataFrame:
    """Applique INTENT_MERGE_MAP et supprime les rows dont l'intent est mappé à None."""
    df = df.copy()
    df["intent"] = df["intent"].map(lambda x: INTENT_MERGE_MAP.get(x, x))
    return df[df["intent"].notna()].reset_index(drop=True)


all_sources = [apply_intent_merge(df) for df in all_sources]

print("Intents après consolidation par source :")
for df in all_sources:
    src = df['source'].iloc[0]
    print(f"  {src:<25} {df['intent'].nunique():>3} intents  |  {len(df):>8,} rows")

Intents après consolidation par source :
  bitext_retail              37 intents  |    44,884 rows
  bitext_support             25 intents  |    26,872 rows
  synthetic_ecommerce         9 intents  |       861 rows
  faq_intents                72 intents  |       394 rows
  amazon_single_qna          18 intents  |   422,635 rows
  amazon_multi_questions     17 intents  |    52,437 rows


## 10. Canonical intent space definition

The canonical intent set is built from `bitext_retail` after consolidation.  
This source is used as the clean reference because its intents are explicit and well defined.

Other sources are kept only when their intents match this canonical space after mapping.


In [10]:
# Use bitext_retail as the stable reference
# Other sources contribute only if their intents match after merging
df_retail_merged = apply_intent_merge(df_retail)

canonical_intents = sorted(df_retail_merged["intent"].dropna().unique().tolist())
canonical_set     = set(canonical_intents)

print(f"Canonical intents : {len(canonical_set)}")
print(canonical_intents)

Canonical intents : 37
['add_product', 'availability', 'cancel_order', 'change_account', 'change_order', 'check_payment_methods', 'check_refund_policy', 'contact_human_agent', 'create_account', 'damaged_delivery', 'delete_account', 'delivery_time', 'exchange_product', 'missing_item', 'order_history', 'pay', 'payment_issue', 'product_information', 'product_issue', 'recover_password', 'remove_product', 'request_invoice', 'request_refund', 'request_right_to_rectification', 'return_product', 'sales_period', 'shipping_costs', 'store_location', 'store_opening_hours', 'submit_feedback', 'submit_product_idea', 'technical_issue', 'track_delivery', 'track_order', 'track_refund', 'use_app', 'wrong_item']


## 11. Canonical filtering

Each source is filtered to keep only samples whose intent belongs to the canonical set.  
A report is generated to show the retention rate per source.


In [11]:
def keep_only_canonical(
    df: pd.DataFrame, canonical_set: set
) -> tuple:
    if df.empty:
        return df, {"source": "empty", "before": 0, "after": 0, "rejected": 0, "kept_rate": 0}
    source_name = df["source"].iloc[0]
    before = len(df)
    kept   = df[df["intent"].isin(canonical_set)].copy()
    after  = len(kept)
    return kept, {
        "source"   : source_name,
        "before"   : before,
        "after"    : after,
        "rejected" : before - after,
        "kept_rate": round(after / before, 4) if before else 0,
    }


filtered_sources = []
filter_reports   = []

for df in all_sources:
    kept, report = keep_only_canonical(df, canonical_set)
    filter_reports.append(report)
    if not kept.empty:
        filtered_sources.append(kept)

filter_report_df = pd.DataFrame(filter_reports)
filter_report_df.to_csv(REPORTS_DIR / "source_filtering_report.csv", index=False)
filter_report_df

,source,before,after,rejected,kept_rate
0,bitext_retail,44884,44884,0,1.0000
1,bitext_support,26872,17959,8913,0.6683
2,synthetic_ecommerce,861,288,573,0.3345
3,faq_intents,394,137,257,0.3477
4,amazon_single_qna,422635,422635,0,1.0000
5,amazon_multi_questions,52437,52437,0,1.0000


## 12. Merge, deduplication, and label encoding

All filtered sources are merged.  
Exact duplicates are removed using the pair `(instruction, intent)`.  
Then intents are encoded into numerical labels for DistilBERT or other classifiers.


In [12]:
df_all = pd.concat(filtered_sources, ignore_index=True)

# Final cleaning
df_all = df_all[df_all["instruction"].str.len() > 0].copy()
df_all = df_all[df_all["intent"].isin(canonical_set)].copy()
df_all = df_all.drop_duplicates(subset=["instruction", "intent"]).reset_index(drop=True)

# Stable label encoding
INTENT_LIST = sorted(df_all["intent"].unique())
LABEL2ID = {intent: idx for idx, intent in enumerate(INTENT_LIST)}
ID2LABEL = {idx: intent for intent, idx in LABEL2ID.items()}

df_all["label"] = df_all["intent"].map(LABEL2ID).astype(int)
df_all["word_count"] = df_all["instruction"].str.split().str.len()
df_all["char_count"] = df_all["instruction"].str.len()
df_all["is_external"] = df_all["source"].ne("bitext_retail")
df_all["is_heuristic"] = df_all["mapping_method"].eq("heuristic")


FINAL_COLS = [
    "instruction", "response", "intent", "label",
    "category", "flags", "source", "mapping_method",
    "word_count", "char_count",
    "has_placeholder", "placeholder_count",
    "is_external", "is_heuristic",
]

df_all = df_all[[c for c in FINAL_COLS if c in df_all.columns]]

print("Merged dataset:", df_all.shape)
display(df_all.head())
display(df_all["source"].value_counts().to_frame("n_samples"))
display(df_all["intent"].value_counts().head(20).to_frame("n_samples"))

Merged dataset: (423766, 14)


,instruction,response,intent,label,category,flags,source,mapping_method,word_count,char_count,has_placeholder,placeholder_count,is_external,is_heuristic
0,I got to add an item to the cart,I'll get right on it! I'm here to assist you i...,add_product,0,cart,BL,bitext_retail,explicit_intent,9,32,False,0,False,False
1,wanna add fucking products to the basket can h...,I sincerely apologize if you've encountered an...,add_product,0,cart,BCIMQWZ,bitext_retail,explicit_intent,10,53,False,0,False,False
2,i have to add products to the basket i ned help,You bet! I'm here to assist you in adding prod...,add_product,0,cart,BCMQZ,bitext_retail,explicit_intent,11,47,False,0,False,False
3,di like to add products to the cart could i ge...,Indeed! I'm here to assist you in adding produ...,add_product,0,cart,BCILMPQZ,bitext_retail,explicit_intent,13,57,False,0,False,False
4,"I need to add an item to the cart , where do I...",I'll take care of it! I'm here to help you wit...,add_product,0,cart,BCILZ,bitext_retail,explicit_intent,15,53,False,0,False,False


,n_samples
source,
amazon_single_qna,360494
bitext_retail,44674
bitext_support,15611
amazon_multi_questions,2562
synthetic_ecommerce,288
faq_intents,137


,n_samples
intent,
product_information,297745
contact_human_agent,20354
delivery_time,17515
track_delivery,17237
availability,9025
damaged_delivery,6548
return_product,4585
recover_password,3542
check_refund_policy,3164


## 13. Source capping and balancing

Very large sources, especially Amazon QA, can dominate the training distribution.  
If this happens, the model may learn the Amazon question style instead of robust conversational intent patterns.

**Solution**: cap large sources with stratified sampling by intent.  
Bitext remains the reference source, while external sources improve generalization without dominating the dataset.


In [13]:
# None = keep all samples, int = maximum number of samples for this source
SOURCE_CAPS = {
    "bitext_retail"         : None,   # keep all samples — reference source
    "bitext_support"        : None,   # keep all samples — high-quality intents
    "amazon_single_qna"     : 30_000, # aggressive cap: prevents Amazon domination
    "amazon_multi_questions": 15_000, # moderate cap
    "synthetic_ecommerce"   : None,   # small source, keep all samples
    "faq_intents"           : None,   # small source, keep all samples
}


def cap_source(df: pd.DataFrame, cap: int | None, seed: int = 42) -> pd.DataFrame:
    """Stratified sampling by intent to preserve class distribution."""
    if cap is None or len(df) <= cap:
        return df
    try:
        return (
            df.groupby("label", group_keys=False)
              .apply(lambda x: x.sample(frac=cap / len(df), random_state=seed))
              .reset_index(drop=True)
        )
    except Exception:
        return df.sample(n=cap, random_state=seed).reset_index(drop=True)


capped_sources = []
total_before   = len(df_all)

for source_name, cap in SOURCE_CAPS.items():
    subset = df_all[df_all["source"] == source_name].copy()
    if subset.empty:
        continue
    capped = cap_source(subset, cap)
    capped_sources.append(capped)
    cap_str = f"{cap:,}" if cap else "tout"
    print(f"  {source_name:<25} {len(subset):>7,} → {len(capped):>7,}  (cap={cap_str})")

# Sources non listées dans SOURCE_CAPS : garder tout
listed_sources = set(SOURCE_CAPS.keys())
unlisted = df_all[~df_all["source"].isin(listed_sources)]
if not unlisted.empty:
    for src in unlisted["source"].unique():
        sub = unlisted[unlisted["source"] == src]
        capped_sources.append(sub)
        print(f"  {src:<25} {len(sub):>7,} → {len(sub):>7,}  (non listé, tout gardé)")

df_all = pd.concat(capped_sources, ignore_index=True)
print(f"\nTotal avant cap : {total_before:,}")
print(f"Total après cap : {len(df_all):,}")
print("\nDistribution par source :")
print(df_all["source"].value_counts())

  bitext_retail              44,674 →  44,674  (cap=tout)
  bitext_support             15,611 →  15,611  (cap=tout)
  amazon_single_qna         360,494 →  30,000  (cap=30,000)
  amazon_multi_questions      2,562 →   2,562  (cap=15,000)
  synthetic_ecommerce           288 →     288  (cap=tout)
  faq_intents                   137 →     137  (cap=tout)

Total avant cap : 423,766
Total après cap : 93,272

Distribution par source :
source
bitext_retail             44674
amazon_single_qna         30000
bitext_support            15611
amazon_multi_questions     2562
synthetic_ecommerce         288
faq_intents                 137
Name: count, dtype: int64


## 14. Source-aware train / validation / test split

The split is no longer Bitext-only.

**Strategy**:
- Split each source separately.
- Stratify by `label` when possible.
- Keep all sources represented in train, validation, and test.
- Use validation to monitor generalization during training.
- Use test to evaluate both Bitext and external-source robustness.

Default ratio:
- 70% train
- 15% validation
- 15% test


In [14]:
def stratified_split_one_source(
    df: pd.DataFrame,
    train_size: float = 0.70,
    val_size: float = 0.15,
    test_size: float = 0.15,
    seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Splits one source into train/validation/test.

    Stratification is applied by label when each label has enough samples.
    If a source is too small or too sparse, the function falls back to random splitting.
    """
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if abs(train_size + val_size + test_size - 1.0) > 1e-6:
        raise ValueError("train_size + val_size + test_size must equal 1.0")

    if len(df) < 10:
        # Very small source: keep it in train only to avoid unstable validation/test samples.
        return df.copy(), pd.DataFrame(), pd.DataFrame()

    label_counts = df["label"].value_counts()
    can_stratify_first = label_counts.min() >= 2 and df["label"].nunique() > 1

    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"] if can_stratify_first else None,
        random_state=seed,
    )

    # Validation size relative to the remaining train_val part.
    val_relative_size = val_size / (train_size + val_size)

    label_counts_trainval = train_val_df["label"].value_counts()
    can_stratify_second = label_counts_trainval.min() >= 2 and train_val_df["label"].nunique() > 1

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_relative_size,
        stratify=train_val_df["label"] if can_stratify_second else None,
        random_state=seed,
    )

    return train_df, val_df, test_df


train_parts, val_parts, test_parts = [], [], []

for source_name, source_df in df_all.groupby("source"):
    tr, va, te = stratified_split_one_source(
        source_df,
        train_size=0.70,
        val_size=0.15,
        test_size=0.15,
        seed=RANDOM_SEED,
    )
    train_parts.append(tr)
    val_parts.append(va)
    test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts,   ignore_index=True)
test_df  = pd.concat(test_parts,  ignore_index=True)

# Shuffle each split after source-wise splitting.
train_df = train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
val_df   = val_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_df  = test_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Split sizes:")
print("  train:", train_df.shape)
print("  val  :", val_df.shape)
print("  test :", test_df.shape)

print("\nSource distribution by split:")
display(pd.crosstab(train_df["source"], columns="train").join(
    pd.crosstab(val_df["source"], columns="val"), how="outer"
).join(
    pd.crosstab(test_df["source"], columns="test"), how="outer"
).fillna(0).astype(int))






Split sizes:
  train: (65285, 14)
  val  : (13993, 14)
  test : (13994, 14)

Source distribution by split:


col_0,train,val,test
source,,,
amazon_multi_questions,1792,385,385
amazon_single_qna,21000,4500,4500
bitext_retail,31271,6701,6702
bitext_support,10927,2342,2342
faq_intents,95,21,21
synthetic_ecommerce,200,44,44


## 15. Light train-only augmentation

Light augmentation is applied only to external training samples.

It simulates user chat variations:
- abbreviations
- punctuation changes
- occasional uppercase words

Validation and test data are never augmented.


In [18]:
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac
from nlpaug.util import Action
import pandas as pd
import random
SYNONYM_STOPWORDS = {
    # function words
    "i", "my", "the", "a", "an", "is", "it", "to", "do", "me",
    "we", "our", "you", "your", "he", "she", "they", "this", "that",
    "was", "are", "be", "been", "have", "has", "had", "will", "would",
    "can", "could", "should", "may", "might", "shall", "not", "no",
    # intent-critical e-commerce words — NEVER replace these
    # replacing "refund" with "rebate" or "order" with "purchase" 
    # can cross intent boundaries
    "order", "refund", "return", "cancel", "track", "delivery",
    "shipping", "payment", "account", "password", "invoice",
    "product", "item", "store", "receipt", "exchange", "missing",
}
# ── build augmenters once — reusing is faster than recreating ─────────────
aug_syn = naw.SynonymAug(
    aug_src="wordnet",
    aug_p=0.15,
    stopwords=list(SYNONYM_STOPWORDS),
)

aug_bert = naw.ContextualWordEmbsAug(
    model_path="distilbert-base-uncased",
    action="substitute",
    aug_p=0.12,
    device="cpu",
)

aug_typo = nac.KeyboardAug(
    aug_char_p=0.04,
    aug_word_p=0.10,
    include_special_char=False,
    include_numeric=False,
    stopwords=list(SYNONYM_STOPWORDS),
)

aug_delete = naw.RandomWordAug(
    action="delete",
    aug_p=0.08,
    stopwords=list(SYNONYM_STOPWORDS),
)

# ── per-intensity augmenter pipelines ─────────────────────────────────────
# naw.SequentialAug chains multiple augmenters
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac
import nlpaug.flow as naf
from nlpaug.util import Action
import pandas as pd
import random

pipeline_light = naf.Sequential([
    aug_syn,
])

pipeline_medium = naf.Sequential([
    aug_syn,
    aug_typo,
    aug_delete,
])

pipeline_heavy = naf.Sequential([
    aug_typo,
    aug_delete,
])
PIPELINE_MAP = {
    "light" : pipeline_light,
    "medium": pipeline_medium,
    "heavy" : pipeline_heavy,
}

def augment_with_nlpaug(text: str, intensity: str = "medium") -> str:
    """Drop-in replacement for augment_text_light using nlpaug."""
    if not isinstance(text, str) or not text.strip():
        return text
    try:
        pipeline = PIPELINE_MAP.get(intensity, pipeline_medium)
        result   = pipeline.augment(text)
        # nlpaug returns a list
        return result[0] if isinstance(result, list) else result
    except Exception:
        return text   # fallback to original if augmenter fails

# ── source-aware application — same loop as before ────────────────────────
SOURCE_AUG_CONFIG = {
    "bitext_retail"         : ("light",  0.15),
    "bitext_support"        : ("light",  0.15),
    "synthetic_ecommerce"   : ("medium", 0.25),
    "faq_intents"           : ("medium", 0.25),
    "amazon_single_qna"     : ("heavy",  0.30),
    "amazon_multi_questions": ("heavy",  0.30),
}

DEFAULT_AUG_CONFIG = ("medium", 0.20)

aug_parts = []

for source_name, source_df in train_df.groupby("source"):
    base_source = source_name.replace("_aug", "")
    intensity, frac = SOURCE_AUG_CONFIG.get(base_source, DEFAULT_AUG_CONFIG)

    if len(source_df) == 0:
        continue

    sample = source_df.sample(frac=frac, random_state=RANDOM_SEED).copy()
    sample["instruction"]    = sample["instruction"].apply(
        lambda t: augment_with_nlpaug(t, intensity=intensity)
    )
    sample["source"]         = base_source + "_nlpaug"
    sample["mapping_method"] = sample["mapping_method"].astype(str) + f"+nlpaug_{intensity}"
    sample["word_count"]     = sample["instruction"].str.split().str.len()
    sample["char_count"]     = sample["instruction"].str.len()

    aug_parts.append(sample)
    print(f"  {base_source:<25} intensity={intensity:<6}  +{len(sample):,} samples")

if aug_parts:
    aug_df   = pd.concat(aug_parts, ignore_index=True)
    train_df = pd.concat([train_df, aug_df], ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("\nTrain after nlpaug augmentation:", train_df.shape)
display(train_df["source"].value_counts().to_frame("n_samples"))

# ── sanity check ──────────────────────────────────────────────────────────
print("\nnlpaug sanity check:")
test_cases = [
    "I want to track my order",
    "can I get a refund for my damaged product",
    "reset my password please",
]
for t in test_cases:
    print(f"  original : {t}")
    print(f"  light    : {augment_with_nlpaug(t, 'light')}")
    print(f"  medium   : {augment_with_nlpaug(t, 'medium')}")
    print(f"  heavy    : {augment_with_nlpaug(t, 'heavy')}")
    print()

  amazon_multi_questions    intensity=heavy   +538 samples


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_

  amazon_multi_questions_nlpaug intensity=medium  +108 samples
  amazon_single_qna         intensity=heavy   +6,300 samples


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_

  amazon_single_qna_nlpaug  intensity=medium  +1,260 samples


...

  synthetic_ecommerce       intensity=medium  +50 samples
  synthetic_ecommerce_nlpaug intensity=medium  +10 samples

Train after nlpaug augmentation: (94418, 14)


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_

,n_samples
source,
bitext_retail,31271
amazon_single_qna,21000
amazon_single_qna_nlpaug,12600
bitext_support,10927
bitext_retail_nlpaug,9382
bitext_support_nlpaug,3278
amazon_multi_questions,1792
amazon_single_qna_nlpaug_nlpaug,1260
amazon_multi_questions_nlpaug,1076



nlpaug sanity check:
  original : I want to track my order
  light    : I want to track my order
  medium   : I want to track my order
  heavy    : I to track my order

  original : can I get a refund for my damaged product
  light    : can I get a refund for my damaged product
  medium   : can I get a refund for my damaged product
  heavy    : can I get a refund for my product

  original : reset my password please
  light    : reset my password please
  medium   : reset my password please
  heavy    : my password please



[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Nour\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_

## 16. Final split validation

This section checks:
- split sizes
- intent coverage
- missing values
- label consistency
- source distribution

A very low `min_per_intent` in train indicates an under-represented intent that should be monitored.


In [19]:
def split_report(df: pd.DataFrame, split_name: str) -> dict:
    counts = df["intent"].value_counts()
    return {
        "split"          : split_name,
        "n_samples"      : len(df),
        "n_intents"      : df["intent"].nunique(),
        "min_per_intent" : int(counts.min()) if len(counts) else 0,
        "max_per_intent" : int(counts.max()) if len(counts) else 0,
        "sources"        : df["source"].nunique(),
    }


split_reports = pd.DataFrame([
    split_report(train_df, "train"),
    split_report(val_df,   "val"),
    split_report(test_df,  "test"),
])
display(split_reports)

print("Missing values:")
for name, sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = sdf[["instruction", "intent", "label", "source"]].isna().sum().sum()
    print(f"  {name}: {missing} missing values")

# Vérifier que val et test ont les mêmes intents que train
train_labels = set(train_df["label"].unique())
val_labels   = set(val_df["label"].unique())
test_labels  = set(test_df["label"].unique())

val_missing  = val_labels  - train_labels
test_missing = test_labels - train_labels

print(f"\nValidation labels not seen in train  : {val_missing  or 'none ✓'}")
print(f"Test labels not seen in train : {test_missing or 'none ✓'}")

,split,n_samples,n_intents,min_per_intent,max_per_intent,sources
0,train,94418,37,828,31317,18
1,val,13993,37,128,4082,6
2,test,13994,37,129,4083,6


Missing values:
  train: 0 missing values
  val: 0 missing values
  test: 0 missing values

Validation labels not seen in train  : none ✓
Test labels not seen in train : none ✓


## 17. Saving the splits

The final splits are saved in CSV format for compatibility with any framework.  
Parquet files are also saved when the required engine is available.


In [20]:
train_df.to_csv(DATA_SPLITS / "train.csv", index=False, encoding="utf-8")
val_df.to_csv(  DATA_SPLITS / "val.csv",   index=False, encoding="utf-8")
test_df.to_csv( DATA_SPLITS / "test.csv",  index=False, encoding="utf-8")

try:
    train_df.to_parquet(DATA_SPLITS / "train.parquet", index=False)
    val_df.to_parquet(  DATA_SPLITS / "val.parquet",   index=False)
    test_df.to_parquet( DATA_SPLITS / "test.parquet",  index=False)
    print("Parquet files saved.")
except Exception as e:
    print("Parquet skipped :", e)

split_reports.to_csv(REPORTS_DIR / "split_report.csv", index=False)

print("Final files :")
for f in ["train.csv", "val.csv", "test.csv"]:
    path = DATA_SPLITS / f
    shape = pd.read_csv(path).shape
    print(f"  - {path} | {shape}")

Parquet files saved.
Final files :
  - D:\conv_nlp_pipeline\data\splits\train.csv | (94418, 14)
  - D:\conv_nlp_pipeline\data\splits\val.csv | (13993, 14)
  - D:\conv_nlp_pipeline\data\splits\test.csv | (13994, 14)


## 18. Final summary

This section gives an overview of the final dataset and displays random training samples.


In [21]:
summary = {
    "Merged dataset (avant cap)"      : len(pd.concat(filtered_sources)),
    "Dataset after capping"                 : len(df_all),
    "Number of intents"                  : df_all["intent"].nunique(),
    "Train samples"                     : len(train_df),
    "Val samples"                       : len(val_df),
    "Test samples"                      : len(test_df),
    "External sources in train"       : sorted(
        train_df[train_df["is_external"] == True]["source"]
        .str.replace("_aug", "").unique().tolist()
    ),
    "Amazon heuristic samples in train": int(train_df["is_heuristic"].sum()),
    "% Amazon in train"               : f"{train_df['is_heuristic'].mean()*100:.1f}%",
}

for k, v in summary.items():
    print(f"{k}: {v}")

print("\nRandom train samples (10 random) :")
display(
    train_df[["instruction", "intent", "source", "mapping_method"]]
    .sample(min(10, len(train_df)), random_state=RANDOM_SEED)
)

Merged dataset (avant cap): 538340
Dataset after capping: 93272
Number of intents: 37
Train samples: 94418
Val samples: 13993
Test samples: 13994
External sources in train: ['amazon_multi_questions', 'amazon_multi_questions_nlpaug', 'amazon_multi_questions_nlpaug_nlpaug', 'amazon_single_qna', 'amazon_single_qna_nlpaug', 'amazon_single_qna_nlpaug_nlpaug', 'bitext_support', 'bitext_support_nlpaug', 'bitext_support_nlpaug_nlpaug', 'faq_intents', 'faq_intents_nlpaug', 'faq_intents_nlpaug_nlpaug', 'synthetic_ecommerce', 'synthetic_ecommerce_nlpaug', 'synthetic_ecommerce_nlpaug_nlpaug']
Amazon heuristic samples in train: 37836
% Amazon in train: 40.1%

Random train samples (10 random) :


,instruction,intent,source,mapping_method
39898,how do I add some articles to purchase [ORDER]?,change_order,bitext_support,explicit_intent
28137,editting details on gold account,change_account,bitext_support,explicit_intent
94362,where can I buy a manual for a icd -sx712 didi...,track_order,amazon_single_qna,heuristic
16980,I need to swap an item of order [ORDER],change_order,bitext_support,explicit_intent
18846,where can i get the invoices from {{Person Name}},request_invoice,bitext_support_nlpaug,explicit_intent+nlpaug_light
41013,"I don't want my profile , I want to cancel it",delete_account,bitext_retail,explicit_intent
12506,I have a holelite Z625cd string trimmer. Will ...,product_information,amazon_single_qna,heuristic
30874,will an iphone 6 pus fit in this pouch?,product_information,amazon_single_qna,heuristic
35293,will foil aork bg 2028 shaver?,product_information,amazon_single_qna_nlpaug,heuristic+nlpaug_heavy
76273,id like to check were my fucking package is ca...,track_delivery,bitext_retail_nlpaug,explicit_intent+nlpaug_light
